In [87]:
%pip install numpy pandas sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [88]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import random

In [89]:
np.random.seed(42)

num_samples = 8000

data = {
    "amount": np.random.exponential(scale=3000, size=num_samples),
    "account_age": np.random.uniform(0, 20, num_samples),
    "transaction_hour": np.random.randint(0, 24, num_samples),
    "num_prev_transactions": np.random.poisson(7, num_samples),
    "is_international": np.random.binomial(1, 0.2, num_samples),
    "device_risk_score": np.random.uniform(0, 1, num_samples),
    "merchant_risk_score": np.random.uniform(0, 1, num_samples)
}

In [90]:
df = pd.DataFrame(data)
df.head()

,amount,account_age,transaction_hour,num_prev_transactions,is_international,device_risk_score,merchant_risk_score
0,1407.804270,14.405358,16,3,0,0.538153,0.480242
1,9030.364293,13.745660,2,4,1,0.604833,0.658260
2,3950.237081,1.915084,22,14,0,0.057004,0.258470
3,2738.827661,18.451448,3,6,0,0.289548,0.256222
4,508.874611,11.369444,1,4,0,0.497113,0.038916


In [91]:
# Base fraud probability
fraud_prob = (
    0.4 * (df["amount"] > 5000) +
    0.3 * (df["is_international"] == 1) +
    0.3 * (df["device_risk_score"] > 0.8) +
    0.2 * (df["merchant_risk_score"] > 0.7)
)

df["fraud"] = (np.random.rand(num_samples) < fraud_prob).astype(int)

print("Fraud Ratio:", df["fraud"].mean())

Fraud Ratio: 0.251875


In [92]:
features = df.columns.drop("fraud")

scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

In [93]:
#Split into Banks (Non-IID)
def split_non_iid(df, num_clients):
    df_sorted = df.sort_values("amount").reset_index(drop=True)
    splits = []
    chunk = len(df_sorted) // num_clients

    for i in range(num_clients):
        splits.append(df_sorted.iloc[i*chunk:(i+1)*chunk])

    return splits

clients_data = split_non_iid(df, 6)

In [94]:
class Client:
    def __init__(self, data):
        self.data = data
        self.size = len(data)
        self.fraud_count = data["fraud"].sum()
        self.fraud_ratio = self.fraud_count / self.size

    def local_train(self, global_W, global_b, lr=0.05, epochs=40, reg=0.01):
        X = self.data[features].values
        y = self.data["fraud"].values

        W = global_W.copy()
        b = global_b
        n = len(y)

        pos_weight = (n - y.sum()) / (y.sum() + 1e-8)

        for epoch in range(epochs):
            z = np.dot(X, W) + b
            y_pred = 1 / (1 + np.exp(-z))

            error = y_pred - y
            error[y == 1] *= pos_weight

            dW = (1/n) * np.dot(X.T, error) + reg * W
            db = (1/n) * np.sum(error)

            current_lr = lr / (1 + 0.01 * epoch)

            W -= current_lr * dW
            b -= current_lr * db

        return W, b

In [95]:
#Logistic Regression Model
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def initialize_model(n_features):
    W = np.random.randn(n_features)
    b = 0
    return W, b

In [96]:
#Local Trining
def local_train(X, y, W, b, lr=0.1, epochs=50):
    n = len(y)
    for _ in range(epochs):
        y_pred = sigmoid(np.dot(X, W) + b)

        dW = (1/n) * np.dot(X.T, (y_pred - y))
        db = (1/n) * np.sum(y_pred - y)

        W -= lr * dW
        b -= lr * db

    return W, b

In [97]:
#Weighted FedAvg
def weighted_fedavg(updates, sizes):
    total = sum(sizes)

    new_W = np.zeros_like(updates[0][0])
    new_b = 0

    for (W, b), size in zip(updates, sizes):
        weight = size / total
        new_W += weight * W
        new_b += weight * b

    return new_W, new_b

In [98]:
#Global Evaluate
from sklearn.metrics import accuracy_score

def evaluate(W, b, df):
    X = df[features].values
    y = df["fraud"].values

    z = np.dot(X, W) + b
    y_prob = 1 / (1 + np.exp(-z))
    y_pred = (y_prob > 0.5).astype(int)

    if len(np.unique(y)) > 1:
        roc = roc_auc_score(y, y_prob)
    else:
        roc = 0.5

    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    acc = accuracy_score(y, y_pred)

    # New Metrics
    predicted_fraud = y_pred.sum()
    false_positives = np.sum((y_pred == 1) & (y == 0))

    return roc, precision, recall, f1, acc, predicted_fraud, false_positives

In [99]:
clients = [Client(data) for data in clients_data]

global_W = np.random.randn(len(features))
global_b = 0

rounds = 15
dropout_rate = 0.2

for r in range(rounds):
    updates = []
    sizes = []
    dropped_clients = 0

    print(f"\n===== Round {r+1} =====")

    for i, client in enumerate(clients):
        print(f"Client {i+1}")
        print(f"  Total Transactions: {client.size}")
        print(f"  Fraud Count: {client.fraud_count}")
        print(f"  Fraud Ratio: {client.fraud_ratio:.4f}")

        if random.random() < dropout_rate:
            print("Dropped This Round")
            dropped_clients += 1
            continue

        W_local, b_local = client.local_train(global_W, global_b)
        updates.append((W_local, b_local))
        sizes.append(client.size)

    print("\nTotal Dropped Clients:", dropped_clients)

    if len(updates) == 0:
        print("All clients dropped. Skipping round.")
        continue

    global_W, global_b = weighted_fedavg(updates, sizes)

    roc, precision, recall, f1, acc, predicted_fraud, false_pos = evaluate(global_W, global_b, df)

    print("\n--- Global Evaluation ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC-AUC: {roc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"Predicted Fraud Count: {predicted_fraud}")
    print(f"False Positives: {false_pos}")


===== Round 1 =====
Client 1
  Total Transactions: 1333
  Fraud Count: 242
  Fraud Ratio: 0.1815
Dropped This Round
Client 2
  Total Transactions: 1333
  Fraud Count: 250
  Fraud Ratio: 0.1875
Client 3
  Total Transactions: 1333
  Fraud Count: 240
  Fraud Ratio: 0.1800
Client 4
  Total Transactions: 1333
  Fraud Count: 225
  Fraud Ratio: 0.1688
Client 5
  Total Transactions: 1333
  Fraud Count: 276
  Fraud Ratio: 0.2071
Client 6
  Total Transactions: 1333
  Fraud Count: 781
  Fraud Ratio: 0.5859

Total Dropped Clients: 1

--- Global Evaluation ---
Accuracy: 0.5870
ROC-AUC: 0.4179
Precision: 0.2152
Recall: 0.2417
F1-score: 0.2277
Predicted Fraud Count: 2263
False Positives: 1776

===== Round 2 =====
Client 1
  Total Transactions: 1333
  Fraud Count: 242
  Fraud Ratio: 0.1815
Dropped This Round
Client 2
  Total Transactions: 1333
  Fraud Count: 250
  Fraud Ratio: 0.1875
Dropped This Round
Client 3
  Total Transactions: 1333
  Fraud Count: 240
  Fraud Ratio: 0.1800
Client 4
  Total Trans